In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
#importing cleaned data
data = pd.read_csv('cleaned_data')

In [3]:
Peak_name = data['peak_name'].value_counts().reset_index()

In [4]:
Result  = pd.DataFrame(columns=["Peak_name"]).copy()

In [5]:
Result['Peak_name'] = Peak_name['peak_name']

In [6]:
Result = Result.dropna()

In [7]:
death_column =  [(len(data[(data['peak_name'] == s) & (data['success'] == 1) & (data['died'] == 1)])) +
                 
                 (len(data[(data['peak_name'] == s) & (data['died'] == 1)& (data['success'] == 0)])) for s in Result['Peak_name']]

In [8]:
Result['Death'] = death_column

In [9]:
success_column = [len(data[(data['peak_name'] == s) & (data['success'] == 1) & (data['died'] == 0)]) for s in Result['Peak_name']]

In [10]:
Result['Success'] = success_column

In [11]:
failed_column =  [len(data[(data['peak_name'] == s) & (data['success'] == 0) & (data['died'] == 0)]) for s in Result['Peak_name']]

In [12]:
Result['Failed'] = failed_column

In [13]:
Result = Result[(Result['Success'] > 100) & (Result['Death'] > 0)]
Result = Result.sort_values(by='Success', ascending=False).reset_index(drop=True)

In [14]:
climbers_column = Result['Death'] + Result['Success'] + Result['Failed']
Result['Total Climbers'] = climbers_column

In [15]:
# computing death rate each season
death_rate_column = (Result['Death'] / Result['Total Climbers'] *100).round(2)

# assigning death rate column to result
Result['Death Rate'] = death_rate_column  

In [16]:
# computing success rate each season
success_rate_column = (Result['Success'] / Result['Total Climbers'] *100).round(2)

# assigning success rate column to result
Result['Success Rate'] = success_rate_column  

In [17]:
 # computing failed rate each season
Failed_rate_column = (Result['Failed'] / Result['Total Climbers'] *100).round(2)

# assigning failed rate column to result
Result['Failed Rate'] = Failed_rate_column

In [18]:
# 1. Slice the table: Grab rows 0, 1, 2, 3 (The Top 4)
top_4 = Result.iloc[:4].copy()

# 2. Slice the table: Grab everything from row 4 to the end (The Others)
bottom_rows = Result.iloc[4:].copy()

# 3. Sum the raw count columns for the bottom rows
others_data = bottom_rows[['Death', 'Success', 'Failed', 'Total Climbers']].sum()

# 4. Recalculate the percentage rates for the newly combined "Others" group
others_data['Death Rate'] = round((others_data['Death'] / others_data['Total Climbers']) * 100, 2)
others_data['Success Rate'] = round((others_data['Success'] / others_data['Total Climbers']) * 100, 2)
others_data['Failed Rate'] = round((others_data['Failed'] / others_data['Total Climbers']) * 100, 2)

# 5. Add the "Others" label for the peak name
others_data['Peak_name'] = 'Others'

# 6. Convert this new data into a 1-row table and glue it to the bottom of your Top 4
others_row_df = pd.DataFrame([others_data])
Mountains_result = pd.concat([top_4, others_row_df], ignore_index=True)

In [23]:
count_cols = ['Death', 'Success', 'Failed', 'Total Climbers']

# 2. Overwrite those columns with the integer version of themselves
Mountains_result[count_cols] = Mountains_result[count_cols].astype(int)
Mountains_result

,Peak_name,Death,Success,Failed,Total Climbers,Death Rate,Success Rate,Failed Rate
0,Everest,306,9953,11554,21813,1.40,45.63,52.97
1,Ama Dablam,32,4375,3999,8406,0.38,52.05,47.57
2,Cho Oyu,52,3812,5026,8890,0.58,42.88,56.54
3,Manaslu,85,1729,2779,4593,1.85,37.64,60.51
4,Others,355,5044,12991,18390,1.93,27.43,70.64


In [24]:
Mountains_result.to_csv('Mountains_result', index=False)